# Secure AI Workshop — Hands-On Lab `00 · Overview`
#### ⏱ ~5 min to read · whole lab ≈ 45 min

**Companion to the worksheet _"Your Turn: Design a Secure Workflow"._**

You designed a secure workflow on paper. These notebooks let you *run* one.
Each notebook implements one part of the seven-layer architecture from the
talk and maps directly to a worksheet step.

| Notebook | Worksheet step | Architecture layer | ⏱ |
|---|---|---|---|
| `01_Input_Safety_PII` | Step 2 · L3 | Input Safety — PII redaction & injection screening | 7 min |
| `02_RAG_Grounding` | Step 2 · L4 | RAG Knowledge — grounding answers in institutional sources | 8 min |
| `03_Confidence_HITL_Audit` | Step 3 | Output Control + HITL + Audit — the 70% threshold & the 62% test | 10 min |
| `04_Scenario_Lab` | Steps 0–3 | End-to-end pipeline for Scenario A or B | 15 min |

> **No API key required.** A deterministic *mock* LLM stands in for the model
> so everything runs offline. The last cell of each notebook shows how to drop
> in the real Claude client.

## Setup — read this first

**What you need:** Python ≥ 3.9 and Jupyter. The labs use only the Python
**standard library** — there is nothing extra to `pip install` to run them.

```bash
pip install notebook        # or: pip install jupyterlab
jupyter lab                 # then open this folder
```

**How to run a notebook**
- Click a cell and press **`Shift` + `Enter`** to run it and move to the next.
- Run cells **top to bottom**. Each notebook is **self-contained** — you can
  open `01`–`04` in any order, but within a notebook run from the top.
- Re-run a cell anytime; nothing writes to disk or the network.

**Start here → ** run the environment check below, then read the architecture
recap, then open `01_Input_Safety_PII.ipynb`.

In [1]:
# Environment check — run me first.
import sys, platform
ok = sys.version_info >= (3, 9)
print("Python :", platform.python_version(), "✓" if ok else "✗ need >= 3.9")
# every dependency below ships with the standard library:
import re, math, hashlib, json, time, textwrap
from collections import Counter
from dataclasses import dataclass
print("Standard-library imports: OK")
print("\nYou're ready. Run the next cell, then open 01_Input_Safety_PII.ipynb.")
assert ok, "Please use Python 3.9 or newer."

Python : 3.14.0 ✓
Standard-library imports: OK

You're ready. Run the next cell, then open 01_Input_Safety_PII.ipynb.


## The seven-layer architecture (recap)

1. **User Layer** — students, faculty, admin (role-based)
2. **AI Access Gateway** — authentication & routing
3. **Input Safety** — PII detection, prompt-injection prevention  ← `nb 01`
4. **RAG Knowledge** — grounding in institutional knowledge bases  ← `nb 02`
5. **LLM Core** — on-premise inference + uncertainty quantification
6. **Output Control** — hallucination detection, confidence scoring  ← `nb 03`
7. **Audit & Governance** — tamper-evident logging, compliance  ← `nb 03`

The golden rule from the talk: **confident answers flow through; uncertain
ones (below the 70% threshold) pause for a human.**

In [2]:
# Shared mock LLM used across the notebooks.
# Deterministic + offline so the labs always run the same way.
import hashlib, textwrap

def mock_llm(prompt: str, context: str = "") -> dict:
    """Pretend to be an on-prem LLM.

    Returns an answer plus a crude self-reported 'confidence' that rises
    when grounding context is supplied (mirrors how RAG lifts reliability).
    """
    grounded = bool(context.strip())
    # confidence is deterministic from the prompt hash, biased up by grounding
    h = int(hashlib.sha256(prompt.encode()).hexdigest(), 16) % 1000 / 1000.0
    base = 0.45 + 0.25 * h            # 0.45 .. 0.70 ungrounded
    confidence = round(min(0.99, base + (0.22 if grounded else 0.0)), 2)
    if grounded:
        answer = ("Based on the institutional sources provided: "
                  + textwrap.shorten(context, width=220, placeholder=' ...'))
    else:
        answer = ("(ungrounded) A plausible-sounding answer with no cited "
                  "source — exactly the kind of output that can hallucinate.")
    return {"answer": answer, "confidence": confidence, "grounded": grounded}

demo = mock_llm("Summarise the late-submission policy.", context="")
print(demo["answer"])
print("confidence:", demo["confidence"])

(ungrounded) A plausible-sounding answer with no cited source — exactly the kind of output that can hallucinate.
confidence: 0.45


### Optional — swap in the real Claude model

The mock above has the same shape as a real call, so you can replace it:

```python
# pip install anthropic ; export ANTHROPIC_API_KEY=...
from anthropic import Anthropic
client = Anthropic()

def real_llm(prompt, context=""):
    msg = client.messages.create(
        model="claude-opus-4-8",
        max_tokens=512,
        system="Answer ONLY from the provided context. Say 'I don't know' if unsupported.",
        messages=[{"role": "user",
                   "content": f"Context:\n{context}\n\nQuestion: {prompt}"}],
    )
    return {"answer": msg.content[0].text, "confidence": None, "grounded": bool(context)}
```

Ask Claude to *also* return a confidence/uncertainty estimate, or derive one
from token log-probs / self-consistency (see `nb 03`).